In [ ]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

SEED = 42
df = load_dataset("behbudiy/uzbek-sentiment-analysis")["train"].to_pandas()
print(df.shape)
print(df.label.value_counts(normalize=True))

README.md:   0%|          | 0.00/673 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

(10000, 2)
label
0    0.5478
1    0.4522
Name: proportion, dtype: float64


In [ ]:
# %% 2. Yengil normalizatsiya + dublikatlarni tozalash
def norm(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[‘’ʻʼ`´]", "'", s)   # apostrof variantlarini bittaga keltirish
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].astype(str).map(norm)
df = df[df.text.str.len() > 1]

n0 = len(df)
# bir xil matn, har xil label -> ziddiyatli, olib tashlaymiz
conf = df.groupby("text")["label"].nunique()
df = df[~df.text.isin(conf[conf > 1].index)]
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"olib tashlandi: {n0 - len(df)}, qoldi: {len(df)}")

olib tashlandi: 1632, qoldi: 8268


In [ ]:
# %% 3. Split (stratified): 80 / 10 / 10
train_df, tmp = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp.label, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

6614 827 827


In [ ]:
# %% 4. Baseline: char n-gram TF-IDF + Logistic Regression
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train_df.text)
Xte = vec.transform(test_df.text)
clf = LogisticRegression(max_iter=1000, C=3).fit(Xtr, train_df.label)
base_pred = clf.predict(Xte)
print("BASELINE  acc:", accuracy_score(test_df.label, base_pred),
      " macro-F1:", f1_score(test_df.label, base_pred, average="macro"))

BASELINE  acc: 0.9044740024183797  macro-F1: 0.8969109416444577


In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(d):
    return Dataset.from_pandas(d[["text", "label"]].reset_index(drop=True), preserve_index=False)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=128)

dsd = DatasetDict({
    "train": to_ds(train_df), "val": to_ds(val_df), "test": to_ds(test_df)
}).map(tokenize, batched=True)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"acc": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

total_steps = (len(train_df) // 32 + 1) * 3   # 32 = batch size, 3 = epoch
warmup = int(0.1 * total_steps)

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup,
    eval_strategy="epoch",          # eski versiyada: evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=dsd["train"], eval_dataset=dsd["val"],
    processing_class=tok,           # eski versiyada: tokenizer=tok
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
)
trainer.train()

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=621, training_loss=0.36958803493235615, metrics={'train_runtime': 582.3288, 'train_samples_per_second': 34.074, 'train_steps_per_second': 1.066, 'total_flos': 692099242238640.0, 'train_loss': 0.36958803493235615, 'epoch': 3.0})

In [ ]:
# %% 6. Test natijasi + solishtirish
out = trainer.predict(dsd["test"])
pred = out.predictions.argmax(-1)
y = np.array(dsd["test"]["label"])
print("XLM-R     acc:", accuracy_score(y, pred), " macro-F1:", f1_score(y, pred, average="macro"))
print(classification_report(y, pred, digits=3))

XLM-R     acc: 0.9117291414752116  macro-F1: 0.9069189080845225
              precision    recall  f1-score   support

           0      0.927     0.929     0.928       507
           1      0.887     0.884     0.886       320

    accuracy                          0.912       827
   macro avg      0.907     0.907     0.907       827
weighted avg      0.912     0.912     0.912       827



In [ ]:
# %% 7. Xatolar tahlili (README uchun)
err = test_df.reset_index(drop=True).copy()
err["pred"] = pred
err["base_pred"] = base_pred
wrong = err[err.label != err.pred]
print("XLM-R xatolari:", len(wrong), "/", len(err))
print(wrong.sample(min(20, len(wrong)), random_state=SEED)[["text", "label", "pred"]].to_string())

rng = np.random.default_rng(42)
yt, b, x = test_df.label.values, base_pred, pred
d = []
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))
    d.append(f1_score(yt[i], x[i], average="macro") - f1_score(yt[i], b[i], average="macro"))
print(np.percentile(d, [2.5, 97.5]))

XLM-R xatolari: 73 / 827
                                                                                                                                                     text  label  pred
35                                                                                 ipi ingicka ekan ipi qalin bulsa uzoqa boradi sicqonca yaxshi yomonmas      1     0
728                                       o'ylagnimdan ham nozik va kichkina ekan. aslida faqat tepasi uchun olgan edim. shuning uchun ham roziman rahmat      1     0
225                                                                                                         yaxshi narsa ekan . lekin 49000 so'mdan keldi      1     0
8                                                                                        manga yekti endi birkun surdim xozircha yaxshi allergiya bermadi      1     0
271                                                                                                                                tewikcha 

In [ ]:
# %% 8. Saqlash (keyin FastAPI + Docker uchun)
trainer.save_model("uz_sentiment_model")
tok.save_pretrained("uz_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('uz_sentiment_model/tokenizer_config.json',
 'uz_sentiment_model/tokenizer.json')

In [ ]:
import torch
print(torch.cuda.is_available())

False


In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # huggingface.co/settings/tokens dan "Write" token kerak

REPO = "yusuf0902/uz-sentiment-xlmr"
trainer.model.push_to_hub(REPO)
tok.push_to_hub(REPO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ze0x7q5/model.safetensors:   0%|          |  560kB / 1.11GB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpg4ztf1fv/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/yusuf0902/uz-sentiment-xlmr/commit/16f0386c647acf0292e14a29f9404d2b0a745097', commit_message='Upload tokenizer', commit_description='', oid='16f0386c647acf0292e14a29f9404d2b0a745097', pr_url=None, repo_url=RepoUrl('https://huggingface.co/yusuf0902/uz-sentiment-xlmr', endpoint='https://huggingface.co', repo_type='model', repo_id='yusuf0902/uz-sentiment-xlmr'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model="yusuf0902/uz-sentiment-xlmr")
print(clf("yaxshi mahsulot ekan, tavsiya qilaman"))

config.json:   0%|          | 0.00/812 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

[{'label': 'LABEL_1', 'score': 0.992962658405304}]
